While runnable as a notebook, this is more convenient in some ways to run like a script - e.g.:

``pixi run jupyter nbconvert --to python psf_training_metaparameter_scan.ipynb --stdout | pixi run -e gpu python -``

In [1]:
from pathlib import Path

import nirspectroperf
import papermill

In [2]:
output_dir = Path('psf_training_metaparameter_scan')
output_dir.mkdir(exist_ok=True)

psf_model_template_path = nirspectroperf.get_template_notebooks()['train_psf_models']

inspected = papermill.inspect_notebook(psf_model_template_path)
inspected

{'random_key_seed': {'name': 'random_key_seed',
  'inferred_type_name': 'None',
  'default': '42',
  'help': 'can be None to use the time'},
 'template_data_file_fn': {'name': 'template_data_file_fn',
  'inferred_type_name': 'None',
  'default': "'../ngc6791_cals/jw02609010001_03101_00001_nrs1_cal.fits'",
  'help': ''},
 'template_slit_number': {'name': 'template_slit_number',
  'inferred_type_name': 'None',
  'default': '49',
  'help': ''},
 'cache_fn': {'name': 'cache_fn',
  'inferred_type_name': 'None',
  'default': "'psf_cache.npz'",
  'help': 'None to not use the cache'},
 'device_dtype': {'name': 'device_dtype',
  'inferred_type_name': 'None',
  'default': 'jnp.float32',
  'help': ''},
 'psf_oversampling': {'name': 'psf_oversampling',
  'inferred_type_name': 'None',
  'default': '4',
  'help': ''},
 'psf_fov': {'name': 'psf_fov',
  'inferred_type_name': 'None',
  'default': '50',
  'help': ''},
 'npsfs': {'name': 'npsfs',
  'inferred_type_name': 'None',
  'default': '2048',
  'he

# Single parameter variation

For the below, we only vary one parameter at a time, and leave the others at their defaults. So these *exclude* the fiduicial case (i.e., all defaults)

In [3]:
template_dict = {'save_model_fn_pattern': None}
template_dict['template_data_file_fn'] = '../' + eval(inspected['template_data_file_fn']['default'])
parameter_dicts = [template_dict] # first one is fiducial

In [4]:
for nepochs in [20000, 60000]:
    parameter_dicts.append(template_dict | {'nepochs': nepochs})

In [5]:
for batch_size in [1024, 4096]:
    parameter_dicts.append(template_dict | {'batch_size': batch_size})

In [6]:
for loss_meantologweight in [0.1, 0.9, 'progressive']:
    parameter_dicts.append(template_dict | {'loss_meantologweight': loss_meantologweight})

In [7]:
for minlr in [1e-5, 1e-4, 1e-3]:
    for peaklr in [1e-3, 5e-3]:
        parameter_dicts.append(template_dict | {'minlr': minlr, 'peaklr': peaklr})

In [8]:
def nb_name_from_pdict(pdict):
    dsub = {k: v for k, v in pdict.items() if k not in template_dict.keys()}
    subname = 'fiducial'

    if dsub != {}:
        tojoin = []
        for k, v in dsub.items():
            if isinstance(v, float):
                tojoin.append(f'{k}{v:.1e}')
            else:
                tojoin.append(f'{k}{v}')

        rawstr = '_'.join(tojoin)
        subname = rawstr.replace('.','p').replace('-','m')
        
    return 'psf_training_' + subname + '.ipynb'

assert len(set([nb_name_from_pdict(pdict) for pdict in parameter_dicts])) == len(parameter_dicts), "Duplicate names found"

len(parameter_dicts)

14

In [9]:
print('Will make the following notebooks:', [nb_name_from_pdict(pdict) for pdict in parameter_dicts])

Will make the following notebooks: ['psf_training_fiducial.ipynb', 'psf_training_nepochs20000.ipynb', 'psf_training_nepochs60000.ipynb', 'psf_training_batch_size1024.ipynb', 'psf_training_batch_size4096.ipynb', 'psf_training_loss_meantologweight1p0em01.ipynb', 'psf_training_loss_meantologweight9p0em01.ipynb', 'psf_training_loss_meantologweightprogressive.ipynb', 'psf_training_minlr1p0em05_peaklr1p0em03.ipynb', 'psf_training_minlr1p0em05_peaklr5p0em03.ipynb', 'psf_training_minlr1p0em04_peaklr1p0em03.ipynb', 'psf_training_minlr1p0em04_peaklr5p0em03.ipynb', 'psf_training_minlr1p0em03_peaklr1p0em03.ipynb', 'psf_training_minlr1p0em03_peaklr5p0em03.ipynb']


Note that if the PSF cache is not already populated the first notebook will take a lot longer to run.

In [ ]:
donenbs = []

for pdict in parameter_dicts:
    nm = output_dir / nb_name_from_pdict(pdict)
    if nm.exists():
        print('Notebook', nm, 'already exists, skipping.')
        donenbs.append(None)
    else:
        print('Notebook', nm, 'does not exist, executing.')
        nb = papermill.execute_notebook(psf_model_template_path, nm, 
                                        parameters=pdict, 
                                        cwd=output_dir,
                                        request_save_on_cell_execute=True,
                                        autosave_cell_every=30,
                                        progress_bar=True)
        donenbs.append(nb)
    
donenbs